# Full-scale Vecchia approximation parameters

## Packages

In [ ]:
import os
import torch
import gpytorch
import matplotlib.pyplot as plt
from matplotlib import colors
import numpy as np
import json
import gpboost as gpb
import requests
import pandas as pd
import time
from properscoring import crps_gaussian as crps_norm

## Data

In [ ]:
data = pd.read_parquet("https://raw.githubusercontent.com/TimGyger/SpaceTimeGPApprox/refs/heads/main/Data/Real_World_TEMP_PRCP.parquet", 
                       engine="pyarrow")
data["date"] = pd.to_datetime(data["date"])

data_complete = (
    data
    .drop_duplicates(subset=["date", "X", "Y"])
    .loc[lambda df: df["date"] < "2025-09-01"]
)

# Arrange by year, doy, then group by year & doy to assign t = cur_group_id()
data_complete = data_complete.sort_values(["year", "doy"])
# Create a unique group id per (year, doy)
data_complete["t"] = pd.factorize(list(zip(data_complete["year"], data_complete["doy"])))[0] + 1

# Drop unused columns
data_complete = data_complete.drop(columns=["year"])
# Build fixed effects / derived features
df = data_complete.copy()
df["tmax"] = (df["tmax"]-32)*5/9
df["KGZones"] = df["KGZones"].astype("category")
df["sin_t"] = np.sin(2 * np.pi * df["doy"] / 366)
df["cos_t"] = np.cos(2 * np.pi * df["doy"] / 366)
df["sin_t2"] = np.sin(4 * np.pi * df["doy"] / 366)
df["cos_t2"] = np.cos(4 * np.pi * df["doy"] / 366)
zone_dummies = pd.get_dummies(df["KGZones"], prefix="KGZones", drop_first=True).astype(float)
for col in zone_dummies.columns:
    df[f"{col}_sin_t"] = zone_dummies[col] * df["sin_t"]
    df[f"{col}_cos_t"] = zone_dummies[col] * df["cos_t"]
    df[f"{col}_sin_t2"] = zone_dummies[col] * df["sin_t2"]
    df[f"{col}_cos_t2"] = zone_dummies[col] * df["cos_t2"]
df = pd.concat([df, zone_dummies], axis=1)
df["X2"] = df["X"]**2
df["Y2"] = df["Y"]**2
df["XY"] = df["X"] * df["Y"]
df["elevation"] = pd.to_numeric(df["elevation"], errors="coerce")
df["elevation2"] = df["elevation"]**2
df["northness"] = np.cos(np.deg2rad(df["aspect_deg"]))
df["eastness"] = np.sin(np.deg2rad(df["aspect_deg"]))
df["slope_north"] = df["slope_deg"] * df["northness"]
df["slope_east"] = df["slope_deg"] * df["eastness"]
df["elevation_slope"] = df["elevation"] * df["slope_deg"]
df["sqrt_distance_to_sea"] = np.sqrt(df["distance_to_sea"])
df["sqrt_distance_to_sea_elevation"] = df["sqrt_distance_to_sea"] * df["elevation"]
df["sin_t_elev"] = df["sin_t"] * df["elevation"]
df["cos_t_elev"] = df["cos_t"] * df["elevation"]
df["sin_t_dist_sea"] = df["sqrt_distance_to_sea"] * df["sin_t"]
df["cos_t_dist_sea"] = df["sqrt_distance_to_sea"] * df["cos_t"]
df["sin_t_X"] = df["X"] * df["sin_t"]
df["cos_t_X"] = df["X"] * df["cos_t"]
df["sin_t_Y"] = df["Y"] * df["sin_t"]
df["cos_t_Y"] = df["Y"] * df["cos_t"]
df["normals"] = df["DLY-TMAX-NORMAL"]

# Drop columns aspect_deg, DLY-TMAX-NORMAL, doy, distance_to_sea
df = df.drop(columns=["aspect_deg", "DLY-TMAX-NORMAL", "doy", "distance_to_sea", "KGZones", "id", "prcp", "prcp_binary"])

df = df.dropna()
# Split train/test by date, and drop 25% stations randomly from train
np.random.seed(42)

# Extract unique stations (X, Y) prior to 2025‑01‑01
stations = (
    df[df["date"] < "2025-01-01"]
    .drop_duplicates(subset=["X", "Y"])
    .reset_index(drop=True)
)

# Sample 25% of these to remove
stations_to_remove = stations.sample(frac=0.25, random_state=42)



In [ ]:
df.head()

In [ ]:
# Generate last day of each month for 2025
last_days = pd.date_range("2025-01-01", "2025-12-31", freq="M")

# Compute DOY
doys = last_days.dayofyear.tolist()

# Add 366 to shift to "year 2"
doys_shifted = [d + 366 for d in doys]
doys_shifted = [366] + doys_shifted
doys_shifted

## Experiments

### Vecchia (Euclidean)

In [ ]:
cov_pars = [] 
coef_pars = [] 
pred_mu = [] 
pred_var = [] 
Time_vec = []
cov_par = pd.DataFrame([[60, 60, 0.037, 6.4939e-06, 1, 1.5, 1, 1]],
                  columns=["Error_term", "sigma2", "a", "c", "alpha", "nu", "beta", "delta"])
coef = pd.DataFrame([[19] + [0]*(df.shape[1]-3)])
for i in range(366, 572):
    print(i)
    df_train = df[df["date"] <= np.max(data_complete["date"][data_complete["t"] == i])]
    # Use anti-join logic: remove rows whose (X, Y) are in stations_to_remove
    mask = df_train.set_index(["X", "Y"]).index.isin(stations_to_remove.set_index(["X", "Y"]).index)

    df_train_filtered = df_train[~mask].copy()
    coords_train = df_train_filtered[["t", "X", "Y"]].to_numpy()
    y_train = df_train_filtered[["tmax"]].to_numpy().ravel()
    X_train = df_train_filtered.drop(columns=["t", "date", "tmax"]).copy()
    X_train.insert(0, "ones", 1)
    X_train = X_train.to_numpy()
    df_test = df[(df["date"] <= np.max(data_complete["date"][data_complete["t"] == i + 7])) & (df["date"] > np.max(data_complete["date"][data_complete["t"] == i]))]
    coords_test = df_test[["t", "X", "Y"]].to_numpy()
    y_test = df_test[["tmax"]].to_numpy().ravel()
    X_test = df_test.drop(columns=["t", "date", "tmax"]).copy()
    X_test.insert(0, "ones", 1)
    X_test = X_test.to_numpy()
    if i in doys_shifted:
        print("Re-train model")
        start_time = time.time()
        model = gpb.GPModel(gp_coords=coords_train, cov_function="space_time_gneiting", cov_fct_shape = 1.5, 
                            likelihood="gaussian",seed = 2,num_neighbors = 30,vecchia_ordering = "time_random_space",
                            matrix_inversion_method = "cholesky", gp_approx="vecchia")
        model.fit(X = X_train,y = y_train,
                 params={"trace": False, "estimate_cov_par_index": [1,1,1,1,1,0,1,1],
                         "init_cov_pars": cov_par.to_numpy().ravel().tolist(), "init_coef": coef.to_numpy().ravel().tolist()})
        end_time = time.time()
        Time_vec.append(end_time-start_time)
        print(end_time-start_time)
        cov_par = model.get_cov_pars()
        cov_pars.append(cov_par)
        coef = model.get_coef()
        coef_pars.append(coef)
    else:
        print("Re-define model")
        start_time = time.time()
        model = gpb.GPModel(gp_coords=coords_train, cov_function="space_time_gneiting", cov_fct_shape = 1.5, 
                            likelihood="gaussian",seed = 2,num_neighbors = 30,vecchia_ordering = "time_random_space",
                            matrix_inversion_method = "cholesky", gp_approx="vecchia")
        model.fit(X = X_train,y = y_train,
                  params={"maxit": 0,"trace": False, "estimate_cov_par_index": [1,1,1,1,1,0,1,1],
                          "init_cov_pars": cov_par.to_numpy().ravel().tolist(), "init_coef": coef.to_numpy().ravel().tolist()})
        cov_par = model.get_cov_pars()
        coef = model.get_coef()
        end_time = time.time()
        
    pred_Linear_Model = model.predict(X_pred=X_test,gp_coords_pred=coords_test,y=y_train,predict_var=True) 
    pred_mu.append(pred_Linear_Model["mu"])
    pred_var.append(pred_Linear_Model["var"])

### Vecchia (Correlation)

In [ ]:
cov_pars = [] 
coef_pars = [] 
pred_mu = [] 
pred_var = [] 
Time_vec = []
cov_par = pd.DataFrame([[60, 60, 0.037, 6.4939e-06, 1, 1.5, 1, 1]],
                  columns=["Error_term", "sigma2", "a", "c", "alpha", "nu", "beta", "delta"])
coef = pd.DataFrame([[19] + [0]*(df.shape[1]-3)])
for i in range(366, 572):
    print(i)
    df_train = df[df["date"] <= np.max(data_complete["date"][data_complete["t"] == i])]
    # Use anti-join logic: remove rows whose (X, Y) are in stations_to_remove
    mask = df_train.set_index(["X", "Y"]).index.isin(stations_to_remove.set_index(["X", "Y"]).index)

    df_train_filtered = df_train[~mask].copy()
    coords_train = df_train_filtered[["t", "X", "Y"]].to_numpy()
    y_train = df_train_filtered[["tmax"]].to_numpy().ravel()
    X_train = df_train_filtered.drop(columns=["t", "date", "tmax"]).copy()
    X_train.insert(0, "ones", 1)
    X_train = X_train.to_numpy()
    df_test = df[(df["date"] <= np.max(data_complete["date"][data_complete["t"] == i + 7])) & (df["date"] > np.max(data_complete["date"][data_complete["t"] == i]))]
    coords_test = df_test[["t", "X", "Y"]].to_numpy()
    y_test = df_test[["tmax"]].to_numpy().ravel()
    X_test = df_test.drop(columns=["t", "date", "tmax"]).copy()
    X_test.insert(0, "ones", 1)
    X_test = X_test.to_numpy()
    if i in doys_shifted:
        print("Re-train model")
        start_time = time.time()
        model = gpb.GPModel(gp_coords=coords_train, cov_function="space_time_gneiting", cov_fct_shape = 1.5, 
                            likelihood="gaussian",seed = 2,num_neighbors = 30,vecchia_ordering = "time_random_space",
                            matrix_inversion_method = "cholesky", gp_approx="vecchia_correlation_based")
        model.fit(X = X_train,y = y_train,
                 params={"trace": False, "estimate_cov_par_index": [1,1,1,1,1,0,1,1],
                         "init_cov_pars": cov_par.to_numpy().ravel().tolist(), "init_coef": coef.to_numpy().ravel().tolist()})
        end_time = time.time()
        Time_vec.append(end_time-start_time)
        print(end_time-start_time)
        cov_par = model.get_cov_pars()
        cov_pars.append(cov_par)
        coef = model.get_coef()
        coef_pars.append(coef)
    else:
        print("Re-define model")
        start_time = time.time()
        model = gpb.GPModel(gp_coords=coords_train, cov_function="space_time_gneiting", cov_fct_shape = 1.5, 
                            likelihood="gaussian",seed = 2,num_neighbors = 30,vecchia_ordering = "time_random_space",
                            matrix_inversion_method = "cholesky", gp_approx="vecchia_correlation_based")
        model.fit(X = X_train,y = y_train,
                  params={"maxit": 0,"trace": False, "estimate_cov_par_index": [1,1,1,1,1,0,1,1],
                          "init_cov_pars": cov_par.to_numpy().ravel().tolist(), "init_coef": coef.to_numpy().ravel().tolist()})
        cov_par = model.get_cov_pars()
        coef = model.get_coef()
        end_time = time.time()
        
    pred_Linear_Model = model.predict(X_pred=X_test,gp_coords_pred=coords_test,y=y_train,predict_var=True) 
    pred_mu.append(pred_Linear_Model["mu"])
    pred_var.append(pred_Linear_Model["var"])

### FITC (kMeans++)

In [ ]:
cov_pars = [] 
coef_pars = [] 
pred_mu = [] 
pred_var = []
Time_vec = []
cov_par = pd.DataFrame([[60, 60, 0.037, 6.4939e-06, 1, 1.5, 1, 1]],
                  columns=["Error_term", "sigma2", "a", "c", "alpha", "nu", "beta", "delta"])
coef = pd.DataFrame([[19] + [0]*(df.shape[1]-3)])
for i in range(366, 572):
    print(i)
    df_train = df[df["date"] <= np.max(data_complete["date"][data_complete["t"] == i])]
    # Use anti-join logic: remove rows whose (X, Y) are in stations_to_remove
    mask = df_train.set_index(["X", "Y"]).index.isin(stations_to_remove.set_index(["X", "Y"]).index)

    df_train_filtered = df_train[~mask].copy()
    coords_train = df_train_filtered[["t", "X", "Y"]].to_numpy()
    y_train = df_train_filtered[["tmax"]].to_numpy().ravel()
    X_train = df_train_filtered.drop(columns=["t", "date", "tmax"]).copy()
    X_train.insert(0, "ones", 1)
    X_train = X_train.to_numpy()
    df_test = df[(df["date"] <= np.max(data_complete["date"][data_complete["t"] == i + 7])) & (df["date"] > np.max(data_complete["date"][data_complete["t"] == i]))]
    coords_test = df_test[["t", "X", "Y"]].to_numpy()
    y_test = df_test[["tmax"]].to_numpy().ravel()
    X_test = df_test.drop(columns=["t", "date", "tmax"]).copy()
    X_test.insert(0, "ones", 1)
    X_test = X_test.to_numpy()
    if i in doys_shifted:
        print("Re-train model")
        start_time = time.time()
        model = gpb.GPModel(gp_coords=coords_train, cov_function="space_time_gneiting", cov_fct_shape = 1.5, 
                            likelihood="gaussian",seed = 2,num_ind_points = 500,ind_points_selection = "kmeans++",
                            matrix_inversion_method = "cholesky", gp_approx="fitc")
        model.fit(X = X_train,y = y_train,
                 params={"trace": False, "estimate_cov_par_index": [1,1,1,1,1,0,1,1],
                         "init_cov_pars": cov_par.to_numpy().ravel().tolist(), "init_coef": coef.to_numpy().ravel().tolist()})
        end_time = time.time()
        Time_vec.append(end_time-start_time)
        print(end_time-start_time)
        cov_par = model.get_cov_pars()
        cov_pars.append(cov_par)
        coef = model.get_coef()
        coef_pars.append(coef)
    else:
        print("Re-define model")
        start_time = time.time()
        model = gpb.GPModel(gp_coords=coords_train, cov_function="space_time_gneiting", cov_fct_shape = 1.5, 
                            likelihood="gaussian",seed = 2,num_ind_points = 500,ind_points_selection = "kmeans++",
                            matrix_inversion_method = "cholesky", gp_approx="fitc")
        model.fit(X = X_train,y = y_train,
                  params={"maxit": 0,"trace": False, "estimate_cov_par_index": [1,1,1,1,1,0,1,1],
                          "init_cov_pars": cov_par.to_numpy().ravel().tolist(), "init_coef": coef.to_numpy().ravel().tolist()})
        cov_par = model.get_cov_pars()
        coef = model.get_coef()
        end_time = time.time()
        
    pred_Linear_Model = model.predict(X_pred=X_test,gp_coords_pred=coords_test,y=y_train,predict_var=True) 
    pred_mu.append(pred_Linear_Model["mu"])
    pred_var.append(pred_Linear_Model["var"])

### FITC (space-time-separated kMeans++)

In [ ]:
cov_pars = [] 
coef_pars = [] 
pred_mu = [] 
pred_var = [] 
Time_vec = []
cov_par = pd.DataFrame([[60, 60, 0.037, 6.4939e-06, 1, 1.5, 1, 1]],
                  columns=["Error_term", "sigma2", "a", "c", "alpha", "nu", "beta", "delta"])
coef = pd.DataFrame([[19] + [0]*(df.shape[1]-3)])
for i in range(366, 572):
    print(i)
    df_train = df[df["date"] <= np.max(data_complete["date"][data_complete["t"] == i])]
    # Use anti-join logic: remove rows whose (X, Y) are in stations_to_remove
    mask = df_train.set_index(["X", "Y"]).index.isin(stations_to_remove.set_index(["X", "Y"]).index)

    df_train_filtered = df_train[~mask].copy()
    coords_train = df_train_filtered[["t", "X", "Y"]].to_numpy()
    y_train = df_train_filtered[["tmax"]].to_numpy().ravel()
    X_train = df_train_filtered.drop(columns=["t", "date", "tmax"]).copy()
    X_train.insert(0, "ones", 1)
    X_train = X_train.to_numpy()
    df_test = df[(df["date"] <= np.max(data_complete["date"][data_complete["t"] == i + 7])) & (df["date"] > np.max(data_complete["date"][data_complete["t"] == i]))]
    coords_test = df_test[["t", "X", "Y"]].to_numpy()
    y_test = df_test[["tmax"]].to_numpy().ravel()
    X_test = df_test.drop(columns=["t", "date", "tmax"]).copy()
    X_test.insert(0, "ones", 1)
    X_test = X_test.to_numpy()
    if i in doys_shifted:
        print("Re-train model")
        start_time = time.time()
        model = gpb.GPModel(gp_coords=coords_train, cov_function="space_time_gneiting", cov_fct_shape = 1.5, 
                            likelihood="gaussian",seed = 2,num_ind_points = 504,ind_points_selection = "space_time_kmeans++",
                            matrix_inversion_method = "cholesky", gp_approx="fitc")
        model.fit(X = X_train,y = y_train,
                 params={"trace": False, "estimate_cov_par_index": [1,1,1,1,1,0,1,1],
                         "init_cov_pars": cov_par.to_numpy().ravel().tolist(), "init_coef": coef.to_numpy().ravel().tolist()})
        end_time = time.time()
        Time_vec.append(end_time-start_time)
        print(end_time-start_time)
        cov_par = model.get_cov_pars()
        cov_pars.append(cov_par)
        coef = model.get_coef()
        coef_pars.append(coef)
    else:
        print("Re-define model")
        start_time = time.time()
        model = gpb.GPModel(gp_coords=coords_train, cov_function="space_time_gneiting", cov_fct_shape = 1.5, 
                            likelihood="gaussian",seed = 2,num_ind_points = 504,ind_points_selection = "space_time_kmeans++",
                            matrix_inversion_method = "cholesky", gp_approx="fitc")
        model.fit(X = X_train,y = y_train,
                  params={"maxit": 0,"trace": False, "estimate_cov_par_index": [1,1,1,1,1,0,1,1],
                          "init_cov_pars": cov_par.to_numpy().ravel().tolist(), "init_coef": coef.to_numpy().ravel().tolist()})
        cov_par = model.get_cov_pars()
        coef = model.get_coef()
        end_time = time.time()
        
    pred_Linear_Model = model.predict(X_pred=X_test,gp_coords_pred=coords_test,y=y_train,predict_var=True) 
    pred_mu.append(pred_Linear_Model["mu"])
    pred_var.append(pred_Linear_Model["var"])

### VIF

In [ ]:
cov_pars = [] 
coef_pars = [] 
pred_mu = [] 
pred_var = [] 
Time_vec = []
cov_par = pd.DataFrame([[60, 60, 0.037, 6.4939e-06, 1, 1.5, 1, 1]],
                  columns=["Error_term", "sigma2", "a", "c", "alpha", "nu", "beta", "delta"])
coef = pd.DataFrame([[19] + [0]*(df.shape[1]-3)])
for i in range(547, 548):
    print(i)
    df_train = df[df["date"] <= np.max(data_complete["date"][data_complete["t"] == i])]
    # Use anti-join logic: remove rows whose (X, Y) are in stations_to_remove
    mask = df_train.set_index(["X", "Y"]).index.isin(stations_to_remove.set_index(["X", "Y"]).index)
    df_train_filtered = df_train
    #df_train_filtered = df_train[~mask].copy()
    coords_train = df_train_filtered[["t", "X", "Y"]].to_numpy()
    y_train = df_train_filtered[["tmax"]].to_numpy().ravel()
    X_train = df_train_filtered.drop(columns=["t", "date", "tmax"]).copy()
    X_train.insert(0, "ones", 1)
    X_train = X_train.to_numpy()
    df_test = df[(df["date"] <= np.max(data_complete["date"][data_complete["t"] == i + 7])) & (df["date"] > np.max(data_complete["date"][data_complete["t"] == i]))]
    coords_test = df_test[["t", "X", "Y"]].to_numpy()
    y_test = df_test[["tmax"]].to_numpy().ravel()
    X_test = df_test.drop(columns=["t", "date", "tmax"]).copy()
    X_test.insert(0, "ones", 1)
    X_test = X_test.to_numpy()
    if i in doys_shifted:
        print("Re-train model")
        start_time = time.time()
        model = gpb.GPModel(gp_coords=coords_train, cov_function="space_time_gneiting", cov_fct_shape = 1.5, 
                            likelihood="gaussian",seed = 2,num_ind_points = 504,ind_points_selection = "space_time_kmeans++",
                            matrix_inversion_method = "cholesky", gp_approx="full_scale_vecchia_correlation_based",num_neighbors = 30,vecchia_ordering = "random")
        model.fit(X = X_train,y = y_train,
                 params={"trace": True, "estimate_cov_par_index": [1,1,1,1,1,0,1,1],
                         "init_cov_pars": cov_par.to_numpy().ravel().tolist(), "init_coef": coef.to_numpy().ravel().tolist()})
        end_time = time.time()
        Time_vec.append(end_time-start_time)
        print(end_time-start_time)
        cov_par = model.get_cov_pars()
        cov_pars.append(cov_par)
        coef = model.get_coef()
        coef_pars.append(coef)
    else:
        print("Re-define model")
        start_time = time.time()
        model = gpb.GPModel(gp_coords=coords_train, cov_function="space_time_gneiting", cov_fct_shape = 1.5, 
                            likelihood="gaussian",seed = 2,num_ind_points = 504,ind_points_selection = "space_time_kmeans++",
                            matrix_inversion_method = "cholesky", gp_approx="full_scale_vecchia_correlation_based",num_neighbors = 30,vecchia_ordering = "random")
        model.fit(X = X_train,y = y_train,
                  params={"maxit": 0,"trace": False, "estimate_cov_par_index": [1,1,1,1,1,0,0,1],
                          "init_cov_pars": cov_par.to_numpy().ravel().tolist(), "init_coef": coef.to_numpy().ravel().tolist()})
        cov_par = model.get_cov_pars()
        coef = model.get_coef()
        end_time = time.time()
        
    pred_Linear_Model = model.predict(X_pred=X_test,gp_coords_pred=coords_test,y=y_train,predict_var=True) 
    pred_mu.append(pred_Linear_Model["mu"])
    pred_var.append(pred_Linear_Model["var"])

### Linear Model

In [ ]:
cov_pars = [] 
coef_pars = [] 
pred_mu = [] 
pred_var = [] 
Time_vec = []
cov_par = pd.DataFrame([[1,0.000000001, 0.000000001]],
                  columns=["Error_term", "sigma2", "range"])
coef = pd.DataFrame([[19] + [0]*(df.shape[1]-3)])
for i in range(366, 572):
    print(i)
    df_train = df[df["date"] <= np.max(data_complete["date"][data_complete["t"] == i])]
    # Use anti-join logic: remove rows whose (X, Y) are in stations_to_remove
    mask = df_train.set_index(["X", "Y"]).index.isin(stations_to_remove.set_index(["X", "Y"]).index)

    df_train_filtered = df_train[~mask].copy()
    coords_train = df_train_filtered[["t", "X", "Y"]].to_numpy()
    y_train = df_train_filtered[["tmax"]].to_numpy().ravel()
    X_train = df_train_filtered.drop(columns=["t", "date", "tmax"]).copy()
    X_train.insert(0, "ones", 1)
    X_train = X_train.to_numpy()
    df_test = df[(df["date"] <= np.max(data_complete["date"][data_complete["t"] == i + 7])) & (df["date"] > np.max(data_complete["date"][data_complete["t"] == i]))]
    coords_test = df_test[["t", "X", "Y"]].to_numpy()
    y_test = df_test[["tmax"]].to_numpy().ravel()
    X_test = df_test.drop(columns=["t", "date", "tmax"]).copy()
    X_test.insert(0, "ones", 1)
    X_test = X_test.to_numpy()
    if i in doys_shifted:
        print("Re-train model")
        start_time = time.time()
        model = gpb.GPModel(gp_coords=coords_train, cov_function="matern", cov_fct_shape = 1.5, 
                            likelihood="gaussian",seed = 1,cov_fct_taper_range = 0.00000001,
                            matrix_inversion_method = "cholesky", gp_approx="tapering")
        model.fit(X = X_train,y = y_train,
                 params={"trace": False, "lr_cov": 0.00000001,
                         "init_cov_pars": cov_par.to_numpy().ravel().tolist(), "init_coef": coef.to_numpy().ravel().tolist()})
        end_time = time.time()
        Time_vec.append(end_time-start_time)
        print(end_time-start_time)
        cov_par = model.get_cov_pars()
        cov_pars.append(cov_par)
        coef = model.get_coef()
        coef_pars.append(coef)
    else:
        print("Re-define model")
        start_time = time.time()
        model = gpb.GPModel(gp_coords=coords_train, cov_function="matern", cov_fct_shape = 1.5, 
                            likelihood="gaussian",seed = 1,cov_fct_taper_range = 0.00000001,
                            matrix_inversion_method = "cholesky", gp_approx="tapering")
        model.fit(X = X_train,y = y_train,
                  params={"maxit": 0,"trace": False, "lr_cov": 0.00000001,
                          "init_cov_pars": cov_par.to_numpy().ravel().tolist(), "init_coef": coef.to_numpy().ravel().tolist()})
        cov_par = model.get_cov_pars()
        coef = model.get_coef()
        end_time = time.time()
        
    pred_Linear_Model = model.predict(X_pred=X_test,gp_coords_pred=coords_test,y=y_train,predict_var=True) 
    pred_mu.append(pred_Linear_Model["mu"])
    pred_var.append(pred_Linear_Model["var"])